In [8]:
import torch

# Check for NVIDIA GPU (CUDA)
print("CUDA Available:", torch.cuda.is_available())

# Check for Apple Silicon Mac GPU (MPS)
print("MPS Available:", torch.backends.mps.is_available())

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

CUDA Available: True
MPS Available: False
Using device: cuda


In [1]:
import pandas as pd

family_df = pd.read_csv('protein_families.csv')

i = 0
name, accession, match_len, seed_len, type_, clan = family_df.iloc[i]

In [2]:
from Bio import SeqIO
from modules.sequences import split_train_val_test

# read the fasta files
recs = [(r.id, str(r.seq)) for r in SeqIO.parse(f"sequences/{accession}.fasta", "fasta")]

# separate to train, validation, and test set
train_seqs, val_seqs, test_seqs = split_train_val_test(recs)

In [10]:
from modules.qmodel import initialize_random_hqmm

n = match_len * 2 + 1
W = 1
s = 21
p, K = initialize_random_hqmm(n, s, W, 'cpu')

In [11]:
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# init transition and emmission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

# encode the symbols into index for the hmmlearn
symbols = list(emit_matdf.index)            # ^ A C D ... Y $  (taken from the CSV row order)
sym_idx = {s: i for i, s in enumerate(symbols)}
BEGIN = sym_idx["^"]                         # dummy begin symbol, emitted only by M0
END = sym_idx["$"]    

In [12]:
from modules.qmodel import convert_to_torch

K_torch = convert_to_torch(K)

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


In [13]:
import torch
import numpy as np
from pathlib import Path

from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm

# Directory for saved models
model_dir = Path("saved_models/phqmm")
model_dir.mkdir(parents=True, exist_ok=True)

da0 = description_accuracy_hqmm(
    p, K_torch, W, sym_idx, test_seqs
)

results = {
    "untrained": (float(da0.mean()), float(da0.std()))
}
histories = {}
trained_models = {}

settings = {
    "large step (0.1)": 0.1,
    "medium step (0.01)": 0.01,
    "small step (0.001)": 0.001,
}

max_iter = 15
tol = 1e-3
batch_size = 16
alpha = 0.8

for label, tau in settings.items():
    print(f"--- {label}: tau={tau}", flush=True)

    K_initial = {
        key: value.detach().clone()
        for key, value in K_torch.items()
    }

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    # Save training results
    histories[label] = hist
    trained_models[label] = K_trained

    # Save checkpoint immediately after training
    model_path = model_dir / f"{name}_{accession}_hqmm_tau_{tau:g}_alpha_{alpha:g}.pt"
    history_path = model_dir / f"{name}_{accession}_history_tau_{tau:g}_alpha_{alpha:g}.csv"

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "W": W,
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # Align devices before evaluation
    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval, K_eval, W, sym_idx, test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Test DA: {results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

--- large step (0.1): tau=0.1
Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14675.89  train_da_mean 0.0005  val_da_mean 0.0006
  iter   1  batch   1/5  loss 215.4813  tau 0.100000
  iter   1  batch   2/5  loss 188.9673  tau 0.100000
  iter   1  batch   3/5  loss 188.8850  tau 0.100000
  iter   1  batch   4/5  loss 196.5849  tau 0.100000
  iter   1  batch   5/5  loss 205.6229  tau 0.100000
iter   1  train     4.4s  log-lik -13868.52  train_da_mean 0.0533  val_da_mean 0.0506
  iter   2  batch   1/5  loss 193.1405  tau 0.080000
  iter   2  batch   2/5  loss 220.1636  tau 0.080000
  iter   2  batch   3/5  loss 180.7752  tau 0.080000
  iter   2  batch   4/5  loss 185.1348  tau 0.080000
  iter   2  batch   5/5  loss 171.7207  tau 0.080000
iter   2  train     9.1s  log-lik -13066.60  train_da_mean 0.1112  val_da_mean 0.0894
  iter   3  batch   1/5  loss 186.2146  tau 0.064000
  iter   3  batch   2/5  loss 207.4043  tau 0.064000
  iter   3  batch   3/5  loss 181.4

In [14]:

import torch
import numpy as np
from pathlib import Path

from modules.qmodel import (
    initialize_random_hqmm,
    convert_to_torch
)
from modules.qtrain import train_hqmm
from modules.qevaluation import description_accuracy_hqmm
from modules.model import (
    init_transition_mat,
    init_emission_mat
)

# ==========================================
# 1. MODEL CONFIGURATION
# ==========================================

n = match_len * 2 + 1
s = 21  # 20 amino acids + $

# Initialize transition and emission matrices
trans_matdf = init_transition_mat(match_len)
emit_matdf = init_emission_mat(match_len, seed=2)

symbols = list(emit_matdf.index)
sym_idx = {symbol: i for i, symbol in enumerate(symbols)}

BEGIN = sym_idx["^"]
END = sym_idx["$"]

# Random initialization assumes modeled symbol IDs are 1...s
assert BEGIN == 0
assert sorted(
    idx for symbol, idx in sym_idx.items()
    if symbol != "^"
) == list(range(1, s + 1))

# ==========================================
# 2. TRAINING CONFIGURATION
# ==========================================

# Vary the number of Kraus operators
settings = {
    f"w={w}": w
    for w in [1, 2, 3, 5, n]
}

# Fixed optimization parameters
tau = 0.1
alpha = 0.8
max_iter = 15
tol = 1e-3
batch_size = 16

# Results containers
results = {}
histories = {}
trained_models = {}

# Directory for saved models
model_dir = Path("saved_models/phqmm")
model_dir.mkdir(parents=True, exist_ok=True)

# ==========================================
# 3. FIXED INITIAL DENSITY MATRIX
# ==========================================

# Keep p identical across all W configurations
torch.manual_seed(2)

G = (
    torch.randn((n, n), dtype=torch.float32)
    + 1j * torch.randn((n, n), dtype=torch.float32)
).to(torch.complex64)

p = G @ G.conj().T
p = p / torch.trace(p).real

# ==========================================
# 4. TRAINING LOOP
# ==========================================

for label, W in settings.items():

    print(
        f"--- {label}: n={n}, W={W}, tau={tau}",
        flush=True
    )

    # Initialize NEW Kraus operators for each W
    torch.manual_seed(100 + W)

    _, K = initialize_random_hqmm(
        n=n,
        s=s,
        w=W,
        device="cpu"
    )

    K_torch = convert_to_torch(K)

    K_initial = {
        key: value.detach().clone()
        for key, value in K_torch.items()
    }

    # --------------------------------------
    # Evaluate untrained model
    # --------------------------------------

    K_ref = next(iter(K_initial.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    with torch.no_grad():
        da0 = description_accuracy_hqmm(
            p_eval,
            K_initial,
            W,
            sym_idx,
            test_seqs
        )

    results[f"{label} (untrained)"] = (
        float(da0.mean()),
        float(da0.std())
    )

    # --------------------------------------
    # Train HQMM
    # --------------------------------------

    K_trained, hist = train_hqmm(
        p=p,
        K=K_initial,
        W=W,
        sym_idx=sym_idx,
        train_seqs=train_seqs,
        val_seqs=val_seqs,
        test_seqs=test_seqs,
        max_iter=max_iter,
        tol=tol,
        tau=tau,
        alpha=alpha,
        batch_size=batch_size,
        verbose=True,
        device=device
    )

    histories[label] = hist
    trained_models[label] = K_trained

    # --------------------------------------
    # Save checkpoint and history
    # --------------------------------------

    model_path = model_dir / (
        f"{name}_{accession}_hqmm_"
        f"w{W}_tau_{tau:g}_alpha_{alpha:g}.pt"
    )

    history_path = model_dir / (
        f"{name}_{accession}_history_"
        f"w{W}_tau_{tau:g}_alpha_{alpha:g}.csv"
    )

    checkpoint = {
        "K": {
            key: value.detach().cpu().clone()
            for key, value in K_trained.items()
        },
        "p": p.detach().cpu().clone(),
        "n": n,
        "s": s,
        "W": W,
        "tau": tau,
        "alpha": alpha,
        "batch_size": batch_size,
        "max_iter": max_iter,
        "tol": tol,
        "epochs_completed": len(hist) - 1,
    }

    torch.save(checkpoint, model_path)
    hist.to_csv(history_path, index=False)

    print(f"Model saved: {model_path}", flush=True)
    print(f"History saved: {history_path}", flush=True)

    # --------------------------------------
    # Evaluate trained model
    # --------------------------------------

    K_ref = next(iter(K_trained.values()))

    p_eval = p.to(
        device=K_ref.device,
        dtype=K_ref.dtype
    )

    K_eval = {
        key: value.to(
            device=K_ref.device,
            dtype=K_ref.dtype
        )
        for key, value in K_trained.items()
    }

    with torch.no_grad():
        da = description_accuracy_hqmm(
            p_eval,
            K_eval,
            W,
            sym_idx,
            test_seqs
        )

    results[label] = (
        float(da.mean()),
        float(da.std())
    )

    print(
        f"Test DA: {results[label][0]:.4f} "
        f"± {results[label][1]:.4f}",
        flush=True
    )

# ==========================================
# 5. FINAL SUMMARY
# ==========================================

print("\nFinal Test DA Results")

for label, (mean, std) in results.items():
    print(f"{label}: {mean:.4f} ± {std:.4f}")


--- w=1: n=79, W=1, tau=0.1


/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14692.18  train_da_mean 0.0002  val_da_mean -0.0000
  iter   1  batch   1/5  loss 190.7993  tau 0.100000
  iter   1  batch   2/5  loss 211.1598  tau 0.100000
  iter   1  batch   3/5  loss 199.3265  tau 0.100000
  iter   1  batch   4/5  loss 194.2927  tau 0.100000
  iter   1  batch   5/5  loss 195.1328  tau 0.100000
iter   1  train     4.8s  log-lik -13751.47  train_da_mean 0.0623  val_da_mean 0.0552
  iter   2  batch   1/5  loss 176.7338  tau 0.080000
  iter   2  batch   2/5  loss 188.7245  tau 0.080000
  iter   2  batch   3/5  loss 201.1738  tau 0.080000
  iter   2  batch   4/5  loss 178.6464  tau 0.080000
  iter   2  batch   5/5  loss 246.7395  tau 0.080000
iter   2  train     9.7s  log-lik -14050.99  train_da_mean 0.0397  val_da_mean 0.0372
  iter   3  batch   1/5  loss 179.0055  tau 0.064000
  iter   3  batch   2/5  loss 168.6244  tau 0.064000
  iter   3  batch   3/5  loss 213.4781  tau 0.064000
  iter   3 

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14665.46  train_da_mean 0.0002  val_da_mean 0.0003
  iter   1  batch   1/5  loss 233.6837  tau 0.100000
  iter   1  batch   2/5  loss 197.7489  tau 0.100000
  iter   1  batch   3/5  loss 183.0435  tau 0.100000
  iter   1  batch   4/5  loss 198.6252  tau 0.100000
  iter   1  batch   5/5  loss 207.4976  tau 0.100000
iter   1  train     6.4s  log-lik -13705.79  train_da_mean 0.0660  val_da_mean 0.0614
  iter   2  batch   1/5  loss 204.8335  tau 0.080000
  iter   2  batch   2/5  loss 198.7351  tau 0.080000
  iter   2  batch   3/5  loss 194.3655  tau 0.080000
  iter   2  batch   4/5  loss 187.9213  tau 0.080000
  iter   2  batch   5/5  loss 167.3531  tau 0.080000
iter   2  train    12.5s  log-lik -13220.58  train_da_mean 0.1011  val_da_mean 0.0894
  iter   3  batch   1/5  loss 223.1263  tau 0.064000
  iter   3  batch   2/5  loss 201.3090  tau 0.064000
  iter   3  batch   3/5  loss 173.3345  tau 0.064000
  iter   3  

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14668.48  train_da_mean 0.0000  val_da_mean -0.0000
  iter   1  batch   1/5  loss 196.9844  tau 0.100000
  iter   1  batch   2/5  loss 186.5408  tau 0.100000
  iter   1  batch   3/5  loss 183.9426  tau 0.100000
  iter   1  batch   4/5  loss 227.3072  tau 0.100000
  iter   1  batch   5/5  loss 261.5838  tau 0.100000
iter   1  train     9.8s  log-lik -13823.70  train_da_mean 0.0574  val_da_mean 0.0541
  iter   2  batch   1/5  loss 182.3444  tau 0.080000
  iter   2  batch   2/5  loss 193.0242  tau 0.080000
  iter   2  batch   3/5  loss 206.9096  tau 0.080000
  iter   2  batch   4/5  loss 208.4365  tau 0.080000
  iter   2  batch   5/5  loss 181.0768  tau 0.080000
iter   2  train    17.7s  log-lik -13579.60  train_da_mean 0.0749  val_da_mean 0.0708
  iter   3  batch   1/5  loss 171.4528  tau 0.064000
  iter   3  batch   2/5  loss 215.8233  tau 0.064000
  iter   3  batch   3/5  loss 215.8598  tau 0.064000
  iter   3 

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14661.51  train_da_mean 0.0003  val_da_mean 0.0003
  iter   1  batch   1/5  loss 217.2111  tau 0.100000
  iter   1  batch   2/5  loss 191.2430  tau 0.100000
  iter   1  batch   3/5  loss 201.5117  tau 0.100000
  iter   1  batch   4/5  loss 202.2323  tau 0.100000
  iter   1  batch   5/5  loss 217.4383  tau 0.100000
iter   1  train    13.3s  log-lik -13877.37  train_da_mean 0.0539  val_da_mean 0.0529
  iter   2  batch   1/5  loss 212.8756  tau 0.080000
  iter   2  batch   2/5  loss 182.3868  tau 0.080000
  iter   2  batch   3/5  loss 218.4774  tau 0.080000
  iter   2  batch   4/5  loss 177.6082  tau 0.080000
  iter   2  batch   5/5  loss 191.7546  tau 0.080000
iter   2  train    26.4s  log-lik -13703.26  train_da_mean 0.0662  val_da_mean 0.0636
  iter   3  batch   1/5  loss 191.0819  tau 0.064000
  iter   3  batch   2/5  loss 179.8785  tau 0.064000
  iter   3  batch   3/5  loss 215.9089  tau 0.064000
  iter   3  

/home/kyle/github/Profile-Hidden-Quantum-Markov-Model/modules/qmodel.py:100: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  key: torch.tensor(


Training on cuda (dtype=torch.complex64)
iter   0  train     0.0s  log-lik -14666.40  train_da_mean -0.0000  val_da_mean -0.0000
  iter   1  batch   1/5  loss 198.6687  tau 0.100000
  iter   1  batch   2/5  loss 217.9023  tau 0.100000
  iter   1  batch   3/5  loss 199.7411  tau 0.100000
  iter   1  batch   4/5  loss 195.7794  tau 0.100000
  iter   1  batch   5/5  loss 219.8247  tau 0.100000
iter   1  train   161.6s  log-lik -13920.57  train_da_mean 0.0507  val_da_mean 0.0480
  iter   2  batch   1/5  loss 202.7850  tau 0.080000
  iter   2  batch   2/5  loss 215.1376  tau 0.080000
  iter   2  batch   3/5  loss 180.6291  tau 0.080000
  iter   2  batch   4/5  loss 198.8103  tau 0.080000
  iter   2  batch   5/5  loss 185.0901  tau 0.080000
iter   2  train   313.5s  log-lik -13794.05  train_da_mean 0.0595  val_da_mean 0.0564
  iter   3  batch   1/5  loss 191.9917  tau 0.064000
  iter   3  batch   2/5  loss 182.9029  tau 0.064000
  iter   3  batch   3/5  loss 201.9721  tau 0.064000
  iter   3

In [16]:
import pandas as pd
import matplotlib.pyplot as plt

from modules.qplots import (
    plot_da_learning_curve,
    plot_da_bars,
)

from pathlib import Path

try:
    plt.style.use("seaborn-v0_8-darkgrid")
except OSError:
    pass

# Create directory for saved models
results_dir = Path("results/phqmm/w")

# Save training histories
for label, h in histories.items():
    h.to_csv(
        f"q_history_{label.split()[0]}.csv",
        index=False
    )

# Save combined training histories
combined_histories = pd.concat(
    [
        h.assign(step_size=label)
        for label, h in histories.items()
    ],
    ignore_index=True,
)

combined_histories.to_csv(
    results_dir / f"{name}_q_training_histories.csv",
    index=False,
)

# Save final description accuracy results
pd.DataFrame(
    results,
    index=["test_da_mean", "test_da_std"]
).T.to_csv(results_dir / f"{name}_q_da_results.csv")

# Optional reference DA values
references = {}

# Test DA vs GPU training time
plot_da_learning_curve(
    histories,
    references,
    x="train_time",
    out=results_dir / f"{name}_q_da_vs_time.png"
)

# Test DA vs epoch
plot_da_learning_curve(
    histories,
    references,
    x="iter",
    out=results_dir / f"{name}_q_da_vs_epoch.png"
)

# Final DA comparison by step size
plot_da_bars(
    results,
    references,
    out=results_dir / f"{name}_q_da_results.png"
)